### Faiss Vector Store

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

---

#### 문서 로드

In [3]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/KCI_FI003153549_p5.pdf")
documents = loader.load()

#### 문서 분할

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splitted_documents = text_splitter.split_documents(documents)

#### 임베딩 모델(캐싱)

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

underlying_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings,
    store,
    namespace = underlying_embeddings.model
)

#### 임베딩 & FAISS(Facebook AI Similarity Search) 벡터스토어 생성 및 저장

##### Case1. In-memory

In [8]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(splitted_documents, cached_embedder)

##### Case2. 로컬 디스크 저장(기존 파일 삭제 후 저장)

In [33]:
vectorstore = FAISS.from_documents(splitted_documents, cached_embedder)

# 영구적인 파일(persistent file)**로 디스크에 저장
# 기존 폴더에 새로운 인덱스 파일을 덮어쓰기 때문에 중복된 파일이 생성되지 않음(항상 가장 마지막에 저장된 벡터스토어의 파일만 존재)
vectorstore.save_local("./faiss_index")

In [34]:
vectorstore

In [36]:
vectorstore = None

In [37]:
vectorstore

In [ ]:
# 벡터스토어 재로딩
vectorstore = FAISS.load_local(
    "./faiss_index", # 저장된 FAISS 인덱스 폴더의 경로
    cached_embedder,
    allow_dangerous_deserialization=True, #  FAISS 인덱스 내 데이터 역직렬화(deserialization) 허용(신뢰할 수 있는 파일 일 경우)
)

In [40]:
vectorstore

##### Case3. 로컬 디스크 저장(기존 파일이 있을 경우 로드)

In [15]:
vectorstore = None

In [16]:
vectorstore

In [ ]:
FAISS_INDEX_PATH = "./faiss_index"

if os.path.exists(FAISS_INDEX_PATH):
    vectorstore = FAISS.load_local(
        FAISS_INDEX_PATH,
        embedding_model,
        allow_dangerous_deserialization=True,
    )
else:
    # FAISS 벡터스토어 생성 및 저장
    vectorstore = FAISS.from_documents(splitted_documents, embedding_model)
    vectorstore.save_local(FAISS_INDEX_PATH)

In [ ]:
vectorstore

In [22]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"
# query = "Advance RAG 기법이 임상시험 데이터 분석에서 수행하는 주요 역할은 무엇인가요?"
# query = "본 연구에서 Private LLM 성능을 평가하기 위해 사용한 지표 3가지는 무엇인가요?"
# query = "국내에서 LLM을 임상시험에 적용한 대표적인 기관과 그 적용 사례를 2가지 이상 말해보세요."
# query = "ROUGE 평가에서 Private LLM과 ChatGPT의 Recall 값은 각각 얼마였나요?"

##### `similarity_search` 메서드

In [41]:

# results = vectorstore.similarity_search(query, k=3) # k는 유사도 검색에서 반환할 상위 문서 개수(top‑k)
results = vectorstore.similarity_search(query, k=5)

for idx, doc in enumerate(results, start=1):
    print(f"[결과 {idx}]\n" + doc.page_content[:300])
    print("---")

[결과 1]
1.2 Validity of Collected Data
본 연구에서는 의료기기 임상시험에 특화된 Private
수집된 데이터셋은 의료기기 임상시험에 특화된
LLM 접근 방법을 제안한다. 이 접근 방법은 도메인 특화
Private LLM 구축을 위해 도메인 적합성과 다양성, 그리
데이터셋 구축, LLM 모델 튜닝, 도메인 특화 프롬프트 적
고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총
용, 그리고 도메인 특화 기능 구현의 네 가지 핵심 단계로
111,954페이지로 구성된 데이터는 의료기기 임상시험의
구성된다. 각 단계는 의
---
[결과 2]
Accuracy analysis.
인 강의 자료 등
Time Automated processes reduce the time
Efficiency required for analysis.  프로토콜 및 보고서 (25%): 임상시험 프로토콜, CSR
Detailed LLM provides detailed insights to (Clinical Study Report) 템플릿 등
Insights support medical decision-making.
 의료기기 특화 문서 (15%): 의료기기 임상시험 계획
Scalability L
---
[결과 3]
적 접근법이 임상시험에서 어떻게 효과적으로 적용될 수
직접적이고 중요한 영향을 미친다[11].
있는지를 제시하고 있으며, 앞으로 더 많은 사례 연구를
의료기기 임상시험 분야의 도메인 특성에 맞게 튜닝하
통해 LLM 기반 AI의 활용 가능성을 탐구할 계획이다.[15]
기 위해 의료기기 임상시험 전문가로부터 총 158개의 문
서(총 11,954 페이지)를 수집하였다. 수집된 문서는 다음
Table 5. Key Benefits of LLM-Based Analysis in
과 같이 분류된다:
Medical Device Clinical Tr
---
[결과 4]
괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수
를 통해 해당 분야에서 최적의 성능을 달성하도록

---

### Faiss 검색 메서드 작동 방식

1. **쿼리 임베딩**: 사용자가 입력한 `query` 텍스트를 임베딩 모델을 사용하여 벡터로 변환합니다.
2. **유사성 검색**: 변환된 쿼리 벡터와 벡터스토어 내의 모든 문서 벡터 간의 **거리**(Distance) 또는 **유사도**(Similarity)를 계산합니다.
    * FAISS는 기본적으로 **L2(유클리드) 거리**를 사용하며, **점수가 0에 가까울수록** 더 유사한 문서임을 의미합니다.
3. **결과 반환**: 계산된 유사도 점수를 기준으로 상위 $k$개의 문서 덩어리(`Document`)를 정렬하여 반환합니다.

### 검색 메서드 상세 비교

| 특징 | `as_retriever` | `similarity_search` |
| :--- | :--- | :--- |
| **반환 객체** | **`Retriever` (Runnable)** | `List[Document]` (문서 리스트) |
| **핵심 역할** | 검색 **전략**을 정의한 '도구' 생성 | 검색 '행위'를 즉시 수행 |
| **활용 환경** | **LCEL 체인(Chain)** 구성 시 필수 | 즉각적인 결과 확인, 단독 로직 |
| **인터페이스** | `.invoke()`, `.batch()`, `.stream()` 지원 | 메서드 호출 즉시 실행 |
| **추가 설정** | MMR, Score Threshold 등 검색 기법 설정 가능 | 검색 개수($k$) 등 기본 파라미터 위주 |

### `as_retriever()`와 Runnable 객체의 이해

`as_retriever()`를 호출하면 단순히 검색 기능을 가진 객체가 아니라, LangChain의 표준 인터페이스인 **`Runnable`** 객체가 생성됩니다. 

1. **체인 구성의 부품 (`|` 연산자)**:
   - `Runnable`이기 때문에 `retriever | prompt | llm`과 같이 파이프라인(`|`) 연산자를 사용하여 다른 컴포넌트와 유연하게 연결할 수 있습니다.
2. **표준화된 실행 메서드**:
   - **`.invoke(query)`**: 단일 질의에 대한 문서를 검색합니다.
   - **`.batch([q1, q2])`**: 여러 쿼리를 병렬로 처리하여 결과를 반환합니다.
3. **검색 전략의 추상화**:
   - 단순히 가장 가까운 벡터만 찾는 것이 아니라, **MMR(Max Marginal Relevance)** 방식을 선택하여 결과의 다양성을 확보하거나, 특정 점수 이상의 문서만 가져오도록(`score_threshold`) 설정을 미리 박아둘(Encapsulation) 수 있습니다.

#### `as_retriever()` 메서드

In [9]:
# 리트리버 생성
retriever = vectorstore.as_retriever()

In [10]:
retriever

VectorStoreRetriever(tags=['FAISS', 'CacheBackedEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000196872DEAB0>, search_kwargs={})

In [12]:
# 예시 질의
query = "본 연구에서 Private LLM 구축을 위해 수집한 문서의 총 페이지 수와 문서 유형별 비율은 어떻게 되나요?"

In [14]:
results = retriever.invoke(query)

for idx, doc in enumerate(results, start=1):
    print(f"[결과 {idx}]\n" + doc.page_content[:300])
    print("---")

[결과 1]
1.2 Validity of Collected Data
본 연구에서는 의료기기 임상시험에 특화된 Private
수집된 데이터셋은 의료기기 임상시험에 특화된
LLM 접근 방법을 제안한다. 이 접근 방법은 도메인 특화
Private LLM 구축을 위해 도메인 적합성과 다양성, 그리
데이터셋 구축, LLM 모델 튜닝, 도메인 특화 프롬프트 적
고 응용 가능성 측면에서 높은 타당성을 갖추고 있다. 총
용, 그리고 도메인 특화 기능 구현의 네 가지 핵심 단계로
111,954페이지로 구성된 데이터는 의료기기 임상시험의
구성된다. 각 단계는 의
---
[결과 2]
Accuracy analysis.
인 강의 자료 등
Time Automated processes reduce the time
Efficiency required for analysis.  프로토콜 및 보고서 (25%): 임상시험 프로토콜, CSR
Detailed LLM provides detailed insights to (Clinical Study Report) 템플릿 등
Insights support medical decision-making.
 의료기기 특화 문서 (15%): 의료기기 임상시험 계획
Scalability L
---
[결과 3]
적 접근법이 임상시험에서 어떻게 효과적으로 적용될 수
직접적이고 중요한 영향을 미친다[11].
있는지를 제시하고 있으며, 앞으로 더 많은 사례 연구를
의료기기 임상시험 분야의 도메인 특성에 맞게 튜닝하
통해 LLM 기반 AI의 활용 가능성을 탐구할 계획이다.[15]
기 위해 의료기기 임상시험 전문가로부터 총 158개의 문
서(총 11,954 페이지)를 수집하였다. 수집된 문서는 다음
Table 5. Key Benefits of LLM-Based Analysis in
과 같이 분류된다:
Medical Device Clinical Tr
---
[결과 4]
괄하며, 국제 표준과 실제 임상시험 환경에서 발생할 수
를 통해 해당 분야에서 최적의 성능을 달성하도록